# JAX Basics for BLUR

This notebook is a companion to the [JAX Primer guide](../../docs/guides/getting_familiar/01_jax_primer.md).
It provides hands-on examples of the three key JAX features used in BLUR:

1. **JIT Compilation** - Speed up functions by compiling to optimized machine code
2. **vmap Batching** - Run many simulations in parallel efficiently
3. **Automatic Differentiation** - Compute gradients through dynamics for control and optimization

**Prerequisites:** Basic familiarity with NumPy. Reading the JAX Primer guide first is recommended.

## Setup

In [ ]:
# Import BLUR first to ensure float64 precision is enabled
from fmd.simulator import SimplePendulum, simulate
from fmd.simulator.params import PENDULUM_1M

# Now import JAX and other libraries
import jax
import jax.numpy as jnp
from jax import jit, vmap, grad
import numpy as np
import time

# Check JAX configuration
print(f"JAX backend: {jax.default_backend()}")
print(f"Default dtype: {jnp.array(1.0).dtype}")

### A Note on Floating Point Precision

BLUR defaults to **float64** for numerical accuracy in physics simulations. However, consumer NVIDIA GPUs (RTX 4070, 4080, 4090, etc.) have very limited float64 throughput—typically 1/64th the speed of float32. This means you may not see significant GPU speedups with the default settings.

**To use float32 for better GPU performance**, set the environment variable before importing BLUR:

```bash
FMD_USE_FLOAT32=1 uv run jupyter notebook
```

**Tradeoffs:**
- **float64 (default)**: Required for numerical stability in long simulations, quaternion normalization, and accurate gradients for optimization
- **float32**: ~2x faster on consumer GPUs, but may accumulate numerical errors over many timesteps

For the simple examples in this notebook, float32 works fine. The precision matters more for long-duration 6-DOF simulations and optimization algorithms like iLQR.

---

## 1. JIT Compilation

JIT (Just-In-Time) compilation transforms Python functions into optimized machine code.
The first call compiles the function; subsequent calls use the cached compiled version.

Let's see this in action with a simple computation.

### 1.1 Simple Function: Without vs With JIT

In [ ]:
def compute_without_jit(x):
    """A function with repeated operations - not JIT compiled."""
    for _ in range(100):
        x = jnp.sin(x) + jnp.cos(x)
    return x

@jit
def compute_with_jit(x):
    """Same function - JIT compiled."""
    for _ in range(100):
        x = jnp.sin(x) + jnp.cos(x)
    return x

In [ ]:
# Create test data
x = jnp.ones(10000)

# Time without JIT
times_no_jit = []
for _ in range(10):
    start = time.time()
    _ = compute_without_jit(x)
    times_no_jit.append(time.time() - start)
avg_no_jit = np.mean(times_no_jit)
print(f"Without JIT: {avg_no_jit*1000:.2f} ms (avg of 10 runs)")

# Time with JIT - first call includes compilation
start = time.time()
_ = compute_with_jit(x)
first_call = time.time() - start
print(f"\nWith JIT (first call, includes compilation): {first_call*1000:.2f} ms")

# Subsequent calls use cached compilation
times_jit = []
for _ in range(10):
    start = time.time()
    _ = compute_with_jit(x)
    times_jit.append(time.time() - start)
avg_jit = np.mean(times_jit)
print(f"With JIT (cached calls): {avg_jit*1000:.2f} ms (avg of 10 runs)")

print(f"\nSpeedup from JIT: {avg_no_jit / avg_jit:.1f}x")

### 1.2 JIT with BLUR Simulations

BLUR's `simulate()` function is already JIT-compatible. Let's see the timing difference.

In [ ]:
# Create a pendulum
pendulum = SimplePendulum(PENDULUM_1M)
initial_state = jnp.array([0.5, 0.0])  # 0.5 rad angle, 0 velocity

# First simulation call - includes JIT compilation
start = time.time()
result1 = simulate(pendulum, initial_state, dt=0.01, duration=10.0)
first_call_time = time.time() - start
print(f"First simulation (with compilation): {first_call_time*1000:.1f} ms")
print(f"  Simulated {len(result1.times)} timesteps")

# Second call - uses cached compilation
start = time.time()
result2 = simulate(pendulum, initial_state, dt=0.01, duration=10.0)
second_call_time = time.time() - start
print(f"\nSecond simulation (cached): {second_call_time*1000:.2f} ms")

# Average over multiple runs
times = []
for _ in range(20):
    start = time.time()
    _ = simulate(pendulum, initial_state, dt=0.01, duration=10.0)
    times.append(time.time() - start)

print(f"Average (cached): {np.mean(times)*1000:.2f} ms (+/- {np.std(times)*1000:.2f} ms)")
print(f"\nSpeedup from caching: {first_call_time / np.mean(times):.1f}x")

**Key Takeaway:** JIT compilation adds one-time overhead, but cached calls are much faster.
For parameter sweeps and optimization, this overhead is amortized over many calls.

---

## 2. vmap Batching

`jax.vmap` transforms a function that operates on single inputs to one that operates on batches.
This is much faster than Python for-loops because the operations are vectorized and compiled together.

### 2.1 Simple Example: Loop vs vmap

In [ ]:
@jit
def process_single(x):
    """Process a single value."""
    return jnp.sin(x) ** 2 + jnp.cos(x) ** 2

# Create batch of inputs
inputs = jnp.linspace(0, 2 * jnp.pi, 10000)

# Method 1: Python loop (slow)
start = time.time()
results_loop = jnp.array([process_single(x) for x in inputs])
loop_time = time.time() - start
print(f"Python loop: {loop_time*1000:.2f} ms")

# Method 2: vmap (fast)
batched_process = jit(vmap(process_single))

# Warm up
_ = batched_process(inputs)

start = time.time()
results_vmap = batched_process(inputs)
vmap_time = time.time() - start
print(f"vmap batched: {vmap_time*1000:.2f} ms")

print(f"\nSpeedup: {loop_time / vmap_time:.1f}x")
print(f"Results match: {jnp.allclose(results_loop, results_vmap)}")

### 2.2 Batch Simulations with vmap

This is where vmap really shines for BLUR: running many simulations in parallel.

In [ ]:
# Define a function that simulates with a given initial angle
def simulate_with_angle(initial_angle):
    """Run simulation and return final state."""
    initial = jnp.array([initial_angle, 0.0])
    result = simulate(pendulum, initial, dt=0.01, duration=5.0)
    return result.states[-1]  # Return final state [angle, velocity]

# Create batch of initial conditions
n_batch = 100
initial_angles = jnp.linspace(0.1, 1.5, n_batch)
print(f"Running {n_batch} simulations with initial angles from {float(initial_angles[0]):.2f} to {float(initial_angles[-1]):.2f} rad")

In [ ]:
# Method 1: Sequential loop
start = time.time()
results_seq = [simulate_with_angle(angle) for angle in initial_angles]
results_seq = jnp.stack(results_seq)
time_sequential = time.time() - start
print(f"Sequential loop: {time_sequential*1000:.1f} ms")

In [ ]:
# Method 2: Batched with vmap
batched_simulate = jit(vmap(simulate_with_angle))

# Warm up (includes compilation)
_ = batched_simulate(initial_angles)

# Time the batched version
start = time.time()
results_vmap = batched_simulate(initial_angles)
time_vmap = time.time() - start

print(f"Batched with vmap: {time_vmap*1000:.2f} ms")
print(f"\nSpeedup: {time_sequential / time_vmap:.1f}x")
print(f"Output shape: {results_vmap.shape}  # (batch, state_dim)")

### 2.3 Scaling with Batch Size

The speedup from vmap increases with batch size.

In [ ]:
batch_sizes = [10, 25, 50]
times_seq_list = []
times_vmap_list = []

for n in batch_sizes:
    angles = jnp.linspace(0.1, 1.5, n)
    
    # Sequential
    start = time.time()
    _ = [simulate_with_angle(a) for a in angles]
    times_seq_list.append(time.time() - start)
    
    # Batched (with fresh compilation for fair comparison)
    batch_fn = jit(vmap(simulate_with_angle))
    _ = batch_fn(angles)  # Warm up
    
    start = time.time()
    _ = batch_fn(angles)
    times_vmap_list.append(time.time() - start)
    
    speedup = times_seq_list[-1] / times_vmap_list[-1]
    print(f"n={n:3d}: sequential={times_seq_list[-1]*1000:7.1f}ms, vmap={times_vmap_list[-1]*1000:6.2f}ms, speedup={speedup:5.1f}x")

**Key Takeaway:** vmap provides substantial speedups for running many simulations.
This is essential for Monte Carlo analysis, parameter sweeps, and optimization.

---

## 3. Automatic Differentiation (grad)

`jax.grad` computes gradients automatically through any differentiable JAX function.
This enables:
- Sensitivity analysis (how do outputs change with inputs?)
- Gradient-based optimization
- Optimal control (trajectory optimization via autodiff)

### 3.1 Simple Gradient Example

In [ ]:
def quadratic_loss(x):
    """Simple loss function: x^2."""
    return x ** 2

# Compute gradient: d/dx(x^2) = 2x
grad_loss = grad(quadratic_loss)

# Test at different points
for x_val in [1.0, 2.0, 3.0, -2.0]:
    loss_val = quadratic_loss(x_val)
    grad_val = grad_loss(x_val)
    print(f"x={x_val:5.1f}: loss={loss_val:5.1f}, grad={grad_val:5.1f} (expected: {2*x_val:5.1f})")

### 3.2 Gradients Through Dynamics

The real power is computing gradients through entire simulations.
Let's compute how sensitive the final pendulum state is to the initial angle.

In [ ]:
def final_angle_loss(initial_angle):
    """Loss: squared final angle.
    
    This represents a simple goal: minimize the final angle.
    The gradient tells us how to adjust the initial angle to reduce the loss.
    """
    initial = jnp.array([initial_angle, 0.0])
    result = simulate(pendulum, initial, dt=0.01, duration=5.0)
    final_angle = result.states[-1, 0]
    return final_angle ** 2

# Compute gradient function
grad_loss_fn = jit(grad(final_angle_loss))

# Evaluate at different initial angles
print("Sensitivity of final angle loss to initial angle:\n")
for init_angle in [0.1, 0.3, 0.5, 0.7, 1.0]:
    loss = final_angle_loss(init_angle)
    gradient = grad_loss_fn(init_angle)
    print(f"Initial angle={init_angle:.1f} rad: loss={loss:.6f}, gradient={gradient:.4f}")

**Interpreting the gradient:**
- Positive gradient: increasing initial angle increases the loss
- Negative gradient: increasing initial angle decreases the loss
- Magnitude: how sensitive the loss is to small changes

### 3.3 Gradient-Based Optimization

Let's use gradients to find an initial velocity that makes the pendulum reach a target final angle.

In [ ]:
def trajectory_loss(initial_velocity, target_final_angle=0.0):
    """Loss: squared error from target final angle.
    
    Given a fixed initial angle of 0.5 rad, find the initial velocity
    that brings the pendulum to the target final angle.
    """
    initial = jnp.array([0.5, initial_velocity])
    result = simulate(pendulum, initial, dt=0.01, duration=2.0)
    final_angle = result.states[-1, 0]
    return (final_angle - target_final_angle) ** 2

# Gradient of loss w.r.t. initial velocity
grad_fn = jit(grad(trajectory_loss))

# Simple gradient descent
velocity = 0.0  # Initial guess
learning_rate = 0.5

print("Optimizing initial velocity to reach target final angle = 0.0 rad")
print("\nIter  Velocity    Loss        Gradient")
print("-" * 45)

for i in range(15):
    loss = trajectory_loss(velocity)
    g = grad_fn(velocity)
    
    if i % 3 == 0 or i == 14:
        print(f"{i:4d}  {velocity:8.4f}  {loss:10.6f}  {g:10.6f}")
    
    velocity = velocity - learning_rate * g

print(f"\nFinal velocity: {velocity:.4f} rad/s")
print(f"Final loss: {trajectory_loss(velocity):.8f}")

### 3.4 Verify the Optimization Result

In [ ]:
# Run simulation with optimized initial velocity
optimized_initial = jnp.array([0.5, velocity])
result_optimized = simulate(pendulum, optimized_initial, dt=0.01, duration=2.0)

print(f"Initial state: angle={0.5:.2f} rad, velocity={velocity:.4f} rad/s")
print(f"Final state: angle={float(result_optimized.states[-1, 0]):.4f} rad (target: 0.0)")
print(f"Final velocity: {float(result_optimized.states[-1, 1]):.4f} rad/s")

**Key Takeaway:** Automatic differentiation through dynamics enables:
- Computing exact (not numerical) gradients
- Efficient gradient-based optimization
- Sensitivity analysis for understanding system behavior

This is the foundation for optimal control algorithms like direct multiple-shooting OCP used in BLUR.

---

## Summary

| Feature | Purpose | Example Use Case |
|---------|---------|------------------|
| `jax.jit` | Compile functions for speed | Repeated simulation calls |
| `jax.vmap` | Batch operations in parallel | Monte Carlo, parameter sweeps |
| `jax.grad` | Automatic differentiation | Optimization, sensitivity analysis |

**Key points:**
1. JIT compilation has one-time overhead but speeds up repeated calls significantly
2. vmap is much faster than Python loops for batch operations
3. grad enables computing exact gradients through entire simulations

**Next steps:**
- Read the [JAX Primer guide](../../docs/guides/getting_familiar/01_jax_primer.md) for more details
- Continue to [Core Concepts](../../docs/guides/getting_familiar/02_core_concepts.md)
- Explore the other notebooks in `notebooks/`